# FASE B: Procesamiento Multimodal de la información

En esta fase vamos a dedicarnos a lo central del proyecto: el procesamiento de la información de entrada (inputs) que puede tener formato de texto, imagen o audio, para después proporcionar una respuesta adecuada a las necesidades especificadas mediante modelos preentrenados de IA.

**Contenidos**<a id='toc0_'></a>   

[Librerías necesarias y funciones previas](#toc1)      
[Grabadora de voz para Google Colab](#toc2)
[Opciones del asistente](#toc3)
- [Registrar medicamentos mediante imágenes](#toc3_1)


<!-- vscode-jupyter-toc-config
    numbering=false
    anchor=true
    flat=false
    minLevel=1
    maxLevel=6
    /vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1'></a>[0. Librerías necesarias y funciones previas](#toc0_)

In [6]:
# Para grabar AUDIO:
!pip install openai-whisper -q
import whisper
from google.colab import output
from base64 import b64decode

# Para reproducir AUDIO:
!pip install edge-tts -q
import edge_tts
import asyncio
from IPython.display import Audio, display

# Para administrar ARCHIVOS:
import json
import os

# Para administrar IMÁGENES:
import PIL.Image

# Para pasar de VOZ a AUDIO:
from whisper import load_model
model = load_model("small") # Cargamos el modelo Whisper

# Función para generar y reproducir el audio
async def generar_voz(texto):
    # Elegimos la voz: 'es-ES-ElviraNeural' (Mujer, España, muy clara)
    # O 'es-ES-AlvaroNeural' si prefieres hombre.
    VOICE = "es-ES-ElviraNeural"
    OUTPUT_FILE = "respuesta_asistente.mp3"

    communicate = edge_tts.Communicate(texto, VOICE, rate="-10%") # rate="-10%" si la quieres más lenta
    await communicate.save(OUTPUT_FILE)

    # Reproducir en Colab
    display(Audio(OUTPUT_FILE, autoplay=True))

# Forma de ejecutar la función:
# await generar_voz(respuesta_texto)

# Para cargar el HISTORIAL DEL USUARIO:
def cargar_memoria():
    if os.path.exists("memoria_salud.json"):
        with open("memoria_salud.json", "r") as f:
            return json.load(f)
    else:
        # Si no existe, creamos un perfil vacío
        return {"nombre": None, "medicinas": [], "historial": []}

def guardar_memoria(datos):
    with open("memoria_salud.json", "w") as f:
        json.dump(datos, f, indent=4)

# Inicializamos la memoria al empezar así:
# memoria = cargar_memoria()


## <a id='toc2'></a>[1. Grabadora de voz para Google Colab](#toc0_)

Como Colab vive en la nube, no puede acceder directamente al micrófono con Python, por lo que necesitaremos definir una función puente, `grabar_audio`, con la que podremos recabar la información deseada a través del micrófono de los usuarios.

In [3]:
# Código JavaScript para grabar audio desde el navegador
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def grabar_audio(segundos=5):
    print(f"Escuchando durante {segundos} segundos...")
    display(output.eval_js(RECORD_JS))
    audio_b64 = output.eval_js(f"record({segundos*1000})")
    audio_bytes = b64decode(audio_b64.split(',')[1])
    with open("audio_usuario.wav", "wb") as f:
        f.write(audio_bytes)
    return "audio_usuario.wav"

#

## <a id='toc3'></a>[2. Opciones del asistente](#toc0_)

### <a id='toc3_1'></a>[2.1. Registrar medicamentos mediante imágenes](#toc0_)

In [ ]:
# 1. Función para procesar la imagen con Gemini Vision
def analizar_receta(ruta_imagen):
    # Cargamos la imagen
    img = PIL.Image.open(ruta_imagen)

    # Prompt específico para extraer datos estructurados
    prompt = """
    Analiza esta receta médica o caja de medicamento.
    Extrae la información en formato JSON puro, sin texto adicional.
    Usa exactamente estos campos:
    "nombre" (nombre del medicamento),
    "dosis" (ej: 500mg),
    "frecuencia" (ej: cada 8 horas),
    "instrucciones" (ej: después de comer).
    Si no encuentras algún dato, pon "no especificado".
    Responde solo con el objeto JSON o una lista de objetos JSON si hay varios.
    """

    # Usamos Gemini 1.5 Flash (es el mejor y más rápido para visión)
    response = model.generate_content(
        [prompt, img],
        generation_config={"response_mime_type": "application/json"}
    )

    # Convertimos el texto de respuesta a un diccionario de Python
    try:
        nueva_medicacion = json.loads(response.text)
        return nueva_medicacion
    except:
        print("Error al leer el formato de la receta.")
        return None

# 2. Función para actualizar nuestra base de datos JSON
def registrar_en_memoria(nuevos_datos):
    # Cargamos lo que ya tenemos
    memoria = cargar_memoria()

    # Si recibimos una lista (varias medicinas), las añadimos todas
    if isinstance(nuevos_datos, list):
        memoria["medicinas"].extend(nuevos_datos)
    else:
        memoria["medicinas"].append(nuevos_datos)

    # Guardamos los cambios en el archivo .json
    guardar_memoria(memoria)
    return memoria